In [1]:
from run_main import run
import json
import pandas as pd
from IPython.display import Image, JSON, display
import helper_funcs as hf

Start with the worm model optimized for forward locomotion:

In [2]:
indir = 'example_W2D18L'
outdir = indir + "_CO2out"
pfolder = 'notebooks'
inputFolderName = pfolder + "/" + indir
outputFolderName = pfolder + "/" + outdir

Delete the output folder if necessary

In [3]:
hf.delete_subfolder_directory(pfolder, outdir)

False

In [4]:
json_data = hf.get_worm_json(inputFolderName)

Add a simple linearly decreasing concentration environment, with peak at the origin and gradient 0.5. The `environments` object defines one or more spatial chemical environments that sensors can monitor. Each environment has a unique `name`, a source location given by `x_center` and `y_center`, and a `grad_steep` value controlling how quickly concentration changes with distance. Sensors select an environment by referencing its name. Multiple sensors may share the same environment.

In [5]:
json_data=hf.add_environment(json_data, "salt_environment")
print('environments: ' + json.dumps(json_data['environments'], indent=2))

environments: {
  "salt_environment": {
    "name": {
      "value": "salt_environment"
    },
    "x_center": {
      "value": 0.0
    },
    "y_center": {
      "value": 0.0
    },
    "grad_steep": {
      "value": 0.5
    }
  }
}


Add a simple sensor of this environment. The sensor detects whether environmental concentration at the worm’s head is increasing or decreasing. It compares a recent average over `sensor_n` seconds with an earlier average over `sensor_m` seconds. `output_1` represents increasing concentration, while `output_2` represents decreasing concentration. The `weights` connect these outputs to nervous-system cells, and `environment` selects the sensed environment.

In [6]:
json_data=hf.add_sensor(json_data, "salt_environment")
print('sensors: ' + json.dumps(json_data['sensors'], indent=2))

sensors: {
  "sensor_1": {
    "environment": {
      "value": "salt_environment"
    },
    "sensor_m": {
      "value": 2.0
    },
    "sensor_n": {
      "value": 2.0
    },
    "outputs": {
      "message": "Available sensor output names for sensor-to-cell connections",
      "value": [
        {
          "name": "output_1",
          "description": "Positive change in sensed concentration (present average above past average)"
        },
        {
          "name": "output_2",
          "description": "Negative change in sensed concentration (past average above present average)"
        }
      ]
    },
    "weights": {
      "message": "Weights from sensor outputs to Nervous System cells",
      "value": []
    }
  }
}


We wish to make the sensor averaging timescales optimizable. The `add_evotag` function makes any numeric JSON parameter available for optimization by the genetic algorithm. The `keys` list identifies the parameter’s location, such as `["sensors", "sensor_1", "sensor_m"]`. It assigns a unique generated evotag unless a name is supplied, then creates a corresponding active entry in `evolvable_ranges`. Appropriate default optimization limits are selected according to the parameter type. The function returns a modified copy of the JSON dictionary.

In [7]:
keys =["sensors", "sensor_1", "sensor_m"]
json_data=hf.add_evotag(json_data, keys )
keys =["sensors", "sensor_1", "sensor_n"]
json_data=hf.add_evotag(json_data, keys )
print('sensors: ' + json.dumps(json_data['sensors'], indent=2))

sensors: {
  "sensor_1": {
    "environment": {
      "value": "salt_environment"
    },
    "sensor_m": {
      "value": 2.0,
      "evotag": "sensors_sensor_1_sensor_m_0"
    },
    "sensor_n": {
      "value": 2.0,
      "evotag": "sensors_sensor_1_sensor_n_0"
    },
    "outputs": {
      "message": "Available sensor output names for sensor-to-cell connections",
      "value": [
        {
          "name": "output_1",
          "description": "Positive change in sensed concentration (present average above past average)"
        },
        {
          "name": "output_2",
          "description": "Negative change in sensed concentration (past average above present average)"
        }
      ]
    },
    "weights": {
      "message": "Weights from sensor outputs to Nervous System cells",
      "value": []
    }
  }
}


Next we add a new random network for the sensor to connect to. The `add_random_cell_network` function adds a specified number of new interneurons to the JSON nervous system and returns the updated JSON together with their names. Cells are named `Cell_1`, `Cell_2`, and so on, skipping names already in use. Every directed pair of new cells is given a chemical connection with the requested probability, with weights sampled uniformly between `-1` and `1`; reciprocal connections may occur independently. The optional `random_seed` makes the generated network reproducible. The new network is initially isolated from all pre-existing cells. Here we add a fully connected network of size four with connection probability unity.

In [8]:
json_data, cell_names=hf.add_random_cell_network(json_data, 4, 1)
print(cell_names)
print(cell_names[0] +' :' + json.dumps(json_data['nervous_system']['cells'][cell_names[0]], indent=2))

['Cell_1', 'Cell_2', 'Cell_3', 'Cell_4']
Cell_1 :{
  "bias": {
    "value": 0.0
  },
  "cell_class": {
    "value": "interneuron"
  },
  "gain": {
    "value": 1.0
  },
  "state": {
    "value": 0.0
  },
  "tau": {
    "value": 1.0
  }
}


We can examine the connections between the newly added cells. The `get_chemical_connection` function searches for a directed chemical connection from `from_cell` to `to_cell`. The get `get_electrical_connection` function searches for an electrical connection between two cells in either stored order because electrical connections are reciprocal. Both return a copy of the complete connection JSON object when found, including its weight and evotag, or `None` when absent. Neither function modifies the supplied JSON dictionary.

In [16]:
for from_cell in cell_names:
    for to_cell in cell_names:
        chem_con = hf.get_chemical_connection(json_data, from_cell, to_cell)
        elec_con = hf.get_electrical_connection(json_data, from_cell, to_cell)
        if chem_con is not None: print('Chem con: ' + json.dumps(chem_con))
        if elec_con is not None: print('Elec con: ' + json.dumps(elec_con))

Chem con: {"from": "Cell_1", "to": "Cell_2", "weight": {"value": -0.18586913872400035}}
Chem con: {"from": "Cell_1", "to": "Cell_3", "weight": {"value": 0.7254708622094168}}
Chem con: {"from": "Cell_1", "to": "Cell_4", "weight": {"value": 0.07882905559129894}}
Chem con: {"from": "Cell_2", "to": "Cell_1", "weight": {"value": -0.5510865837770966}}
Chem con: {"from": "Cell_2", "to": "Cell_3", "weight": {"value": 0.6601165290498834}}
Chem con: {"from": "Cell_2", "to": "Cell_4", "weight": {"value": -0.12006656785454894}}
Chem con: {"from": "Cell_3", "to": "Cell_1", "weight": {"value": -0.0865608484803293}}
Chem con: {"from": "Cell_3", "to": "Cell_2", "weight": {"value": 0.7451144987158862}}
Chem con: {"from": "Cell_3", "to": "Cell_4", "weight": {"value": 0.34773416134911606}}
Chem con: {"from": "Cell_4", "to": "Cell_1", "weight": {"value": 0.9566970251341098}}
Chem con: {"from": "Cell_4", "to": "Cell_2", "weight": {"value": 0.6080626583471294}}
Chem con: {"from": "Cell_4", "to": "Cell_3", "

We also want to make `add_chemical_connection_evotag` makes the weight of an existing directed chemical connection optimizable by the genetic algorithm. It takes the JSON dictionary and the names of the source and destination cells, then adds a unique evotag to the matching connection weight. A corresponding active entry with suitable default limits is added to `evolvable_ranges`; a custom evotag name may also be supplied. The function returns a modified copy of the JSON and raises an error if the connection does not exist.

In [18]:
for from_cell in cell_names:
    for to_cell in cell_names:
        if from_cell != to_cell:
            json_data = hf.add_chemical_connection_evotag(json_data, from_cell, to_cell)
            chem_con = hf.get_chemical_connection(json_data, from_cell, to_cell)
            if chem_con is not None: print('Chem con: ' + json.dumps(chem_con))

Chem con: {"from": "Cell_1", "to": "Cell_2", "weight": {"value": -0.18586913872400035, "evotag": "ns_chemcons_10"}}
Chem con: {"from": "Cell_1", "to": "Cell_3", "weight": {"value": 0.7254708622094168, "evotag": "ns_chemcons_11"}}
Chem con: {"from": "Cell_1", "to": "Cell_4", "weight": {"value": 0.07882905559129894, "evotag": "ns_chemcons_12"}}
Chem con: {"from": "Cell_2", "to": "Cell_1", "weight": {"value": -0.5510865837770966, "evotag": "ns_chemcons_13"}}
Chem con: {"from": "Cell_2", "to": "Cell_3", "weight": {"value": 0.6601165290498834, "evotag": "ns_chemcons_14"}}
Chem con: {"from": "Cell_2", "to": "Cell_4", "weight": {"value": -0.12006656785454894, "evotag": "ns_chemcons_15"}}
Chem con: {"from": "Cell_3", "to": "Cell_1", "weight": {"value": -0.0865608484803293, "evotag": "ns_chemcons_16"}}
Chem con: {"from": "Cell_3", "to": "Cell_2", "weight": {"value": 0.7451144987158862, "evotag": "ns_chemcons_17"}}
Chem con: {"from": "Cell_3", "to": "Cell_4", "weight": {"value": 0.34773416134911

We also want to make some cell parameters optimizable by the genetic algorithm. The `add_cell_parameter_evotag` function makes a specified nervous-system cell parameter optimizable by the genetic algorithm. It takes the JSON dictionary, cell name, and parameter name, such as `tau` or `bias`. The function adds a unique evotag to that parameter and creates an active entry in `evolvable_ranges` with suitable default limits. A custom evotag name can optionally be supplied. It returns a modified copy of the JSON dictionary.

In [ ]:
evolvable_pars = ["tau", "bias"]
for cell_name in cell_names:
    for evolvable_par in evolvable_pars:
        json_data=hf.add_cell_parameter_evotag(json_data, cell_name, evolvable_par)
print(cell_names[0] +' :' + json.dumps(json_data['nervous_system']['cells'][cell_names[0]], indent=2))